# Chapter 1 · The AI Diet Coach: A Simple Conversational Agent
**Mastering Agentic AI** — Manning Publications

This notebook demonstrates the Perceive → Reason → Respond loop using the simplest possible agent implementation.




## 1.0 · Setup

See `appendix_a_api_keys.md` in this chapter directory for credential setup.

In [ ]:
# !pip install anthropic python-dotenv
import os
from anthropic import Anthropic
from dotenv import load_dotenv
load_dotenv()
assert os.getenv("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY — see appendix_a_api_keys.md"
print("Ready")

## 1.1 · The Perceive → Reason → Respond Loop

The three stages of every agent — even the simplest one:

1. **Perceive** — read the user's message and conversation history
2. **Reason** — LLM forward pass; decide what to say
3. **Respond** — return the reply and update history

What makes this an agent? The LLM autonomously decides what to say and how much to elaborate. That decision-making under uncertainty is the core of agency.

**What is missing at this stage?** No tools (cannot look up real data), no persistent memory (forgets between sessions), no multi-agent coordination. We will add all three in later chapters.

In [ ]:
MODEL = "claude-opus-4-5"
MAX_TOKENS = 1024
HISTORY_LIMIT = 20  # prevents context window overflow

SYSTEM_PROMPT = """You are an AI Diet Coach — a knowledgeable, encouraging nutrition assistant.
You follow the Perceive → Reason → Respond loop in Chapter 1.

Rules (Chapter 1 — no tools yet):
  - Work from training knowledge only; you cannot look up live data.
  - Never fabricate specific calorie counts for branded products.
  - Always recommend consulting a registered dietitian for medical advice."""

print(f"System prompt: {len(SYSTEM_PROMPT)} characters")

## 1.2 · Single-Turn Test

In [ ]:
client = Anthropic()
response = client.messages.create(
    model=MODEL, max_tokens=MAX_TOKENS, system=SYSTEM_PROMPT,
    messages=[{"role": "user", "content": "What should I eat for breakfast to lose weight?"}]
)
print(response.content[0].text)

## 1.3 · Multi-Turn Conversation (In-Context State)

In [ ]:
def coaching_session(messages_in: list[str]) -> list[str]:
    """Simulate a multi-turn session. Returns one reply per input message."""
    history, replies = [], []
    for msg in messages_in:
        history.append({"role": "user", "content": msg})
        response = client.messages.create(
            model=MODEL, max_tokens=MAX_TOKENS, system=SYSTEM_PROMPT,
            messages=history[-HISTORY_LIMIT:]
        )
        reply = response.content[0].text
        history.append({"role": "assistant", "content": reply})
        replies.append(reply)
    return replies

conversation = [
    "Hi, I want to lose 5kg in two months.",
    "I usually skip breakfast. Is that a problem?",
    "What high-protein breakfast takes under 10 minutes?",
]
for q, a in zip(conversation, coaching_session(conversation)):
    print(f"User:  {q}")
    print(f"Coach: {a[:250]}...")
    print("-" * 50)

## 1.4 · What Is Missing?

Observe the coach's limitation — it can only guess about nutrient data.

In [ ]:
response = client.messages.create(
    model=MODEL, max_tokens=MAX_TOKENS, system=SYSTEM_PROMPT,
    messages=[{"role": "user", "content": "How many calories are in 100g of oats?"}]
)
print("Coach (no tools):", response.content[0].text[:350])
print()
print("This answer comes from training data, not a live lookup.")
print("Chapter 2 adds tools so the coach can look up real data.")

## 1.5 · Summary

| What we built | What it demonstrates |
|---|---|
| Single-turn call | Degenerate agent: perceive → reason → respond |
| Multi-turn history | In-context statefulness |
| HISTORY_LIMIT | Token budget management |

**Next:** Chapter 2 adds CrewAI workflows and real tool use.

---
*Mastering Agentic AI · Manning Publications*